# Resonance S21 sweep and timestream generator 
The purpose of this notebook is to outline the resonance S21 generation functions. 

In [ ]:
# This is a work in progress
from citkid.res.generator import get_S21_noise_ts, get_S21_vs_freq, get_S21_vs_freq_dual
import numpy as np
import matplotlib.pyplot as plt 

In [ ]:
### noise parameters
fs = 20_000. # sample frequency
alpha = 1.0 # 1/f noise exponent
f_knee = 10.0 # 1/f noise knee frequency
tau_qp = 1e-3 # QP lifetime
tau_rd = 1e-5 # resonator ringdown time
sxx_white = 1e-15 # white Sxx level in 1 / Hz
sAA_white = 10 ** (-90 / 10) # white SAA level in dBc / Hz

### Resonator parameters
fr = 1e9
Qc = 30e3 
Qi = 300e3 
Qr = 1 / (1 / Qc + 1 / Qi)
amp = Qr / Qc
phi = 0.
a = 0.5
p = np.array([fr, Qr, amp, phi, a])

In [ ]:
### Fine Sweep
nsamps = 10 # number of samples to average per data point

# frequency array for sweeping
span = 5 * fr / Qr 
f = np.linspace(fr - span / 2, fr + span / 2, 500)

zf = get_S21_vs_freq(
    f,
    alpha,
    f_knee,
    tau_qp,
    sxx_white,
    sAA_white,
    fs,
    nsamps,
    p
)

### Timestream 
ft = fr - 20e3 # tone frequency
npoints = 10_000 # number of points

zt = get_S21_noise_ts(
    ft,
    npoints,
    alpha,
    f_knee,
    tau_qp,
    tau_rd,
    sxx_white,
    sAA_white,
    fs,
    p
)

In [ ]:
fig, axs = plt.subplots(
    1, 3, figsize = [15, 4], layout = 'tight',
    gridspec_kw = {'width_ratios': [1, 1, 2]}
)
axs[0].set(ylabel = 'S21 (dB)', xlabel = rf'$f - {fr / 1e6:.02f}$ (kHz)')
axs[1].set(ylabel = 'Q', xlabel = 'I')
axs[2].set(xlabel = 'Time (s)', ylabel = 'S21')

dB = 20 * np.log10(np.abs(zf))
axs[0].plot((f - fr) / 1e3, dB)
axs[1].plot(zf.real, zf.imag, '.', label = 'Sweep')
axs[1].plot(zt.real, zt.imag, '.', label = 'Timestream')
axs[1].legend(loc = 'center')

t = np.arange(zt.size) / fs
axs[2].plot(t, zt.real, label = 'I')
axs[2].plot(t, zt.imag, label = 'Q')
axs[2].legend(loc = 'upper right')


In [ ]:
### Double resonance generation 
fr = 1e9
Qc = 30e3 
Qi = 300e3 
Qr = 1 / (1 / Qc + 1 / Qi)
amp = Qr / Qc
lw = fr / Qr
phi = 0.
a = 0.5
p1 = np.array([fr, Qr, amp, phi, a])
p2 = np.array([fr + 2 * lw, Qr, amp, phi, a])
fl_phase = np.pi / 2


span = 5 * fr / Qr 
f_double = np.linspace(p1[0] - span / 2, p2[0] + span / 2, 500)
zf_double = get_S21_vs_freq_dual(
    f_double,
    alpha,
    f_knee,
    tau_qp,
    sxx_white,
    sAA_white,
    fs,
    nsamps,
    p1,
    p2,
    fl_phase
)

In [ ]:
fig, axs = plt.subplots(
    1, 2, figsize = [8, 4], layout = 'tight'
)
axs[0].set(ylabel = 'S21 (dB)', xlabel = rf'$f - {fr / 1e6:.02f}$ (kHz)')
axs[1].set(ylabel = 'Q', xlabel = 'I')

dB_double = 20 * np.log10(np.abs(zf_double))
axs[0].plot((f_double - fr) / 1e3, dB_double)
axs[1].plot(zf_double.real, zf_double.imag, '.')